In [ ]:
import numpy as np
from pynq import Overlay
import time

In [ ]:
# Load overlay
overlay = Overlay("bnn.bit")
bnn = overlay.bnn_0

In [ ]:
INPUT_BITS = [
    [4294967295, 4294967295, 4294967295, 4294967295, 4294967295, 4294967295, 4290838527, 4227859455, 3221241855, 3758358527, 4232052735, 2281701368, 2147483407, 4294963455, 4294844415, 4293132287, 4232052735, 2281701360, 2147483407, 4294959359, 4294844415, 4290904063, 4229955583, 3288334335, 4294901760],
    [4294967295, 4294967295, 4294967292, 134217600, 1073737731, 4294837311, 4292985855, 4282138623, 4286840831, 4034920447, 268435424, 4294966303, 4294951935, 4294459391, 4278714367, 4034920447, 268435440, 67239680, 2093056, 33521671, 4294967295, 4294967295, 4294967295, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967295, 536870897, 4294967103, 4294960127, 4294852607, 4293132287, 4236247039, 3355443196, 2147483527, 4294965503, 4294938623, 4294049791, 4280287231, 4060086271, 536870881, 4294966815, 4294951935, 4294721535, 4294967295, 4294967295, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967280, 2147483399, 4294959231, 4294705663, 4290777087, 4160815103, 524256, 404749831, 3254771964, 536756161, 4293131295, 4265542143, 3824173054, 1040711648, 16776704, 268427264, 4294901887, 4294707199, 4294967295, 4294967295, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967295, 4294966526, 2147471335, 4294508095, 4287620095, 4043194367, 532938723, 4232052287, 2281687032, 4294737807, 4291031295, 4261416959, 3758161919, 2199912447, 4043309055, 536870897, 4294967071, 4294963711, 4294918143, 4294967295, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967295, 4294967280, 4294966799, 4294959359, 4294713343, 4291035135, 4232052735, 3288334328, 2147483527, 4294965375, 4294905855, 4293984255, 4279238655, 3791650814, 536870881, 4294966815, 4294951935, 4294721535, 4293132287, 4294967295, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967295, 4294965375, 2415857648, 4293000991, 4263502335, 3288088572, 1070071747, 4043308032, 268419072, 4294836255, 4294959615, 4294852607, 4291035135, 4236247039, 3355443192, 4294967177, 4294965279, 4294935039, 4294459391, 4294967295, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967295, 4294967295, 4294959615, 4294713343, 4286644223, 4161011711, 4194289, 33554204, 536867267, 4294901791, 4293919231, 4286648319, 4231069695, 4287102975, 4169138175, 3288334334, 536870881, 4294967071, 4294965503, 4294938623, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967295, 2155872224, 134213632, 2147418119, 4279240703, 3785883644, 536870787, 4294965375, 4294938623, 4294459391, 4286579711, 4160757759, 2147549183, 2148532222, 16777184, 268434944, 4294963263, 4294952959, 4294967295, 4294967295, 4294967295, 4294901760],
    [4294967295, 4294967295, 4294967295, 4294967295, 4294967295, 4294967295, 4294934783, 4292870655, 4160757759, 25231296, 2014313502, 16760832, 536608771, 4293918847, 4294709247, 4286840831, 4030726142, 134217696, 4294966303, 4294935039, 4294459391, 4287102975, 4169138175, 2415919103, 4294901760]
]

EXPECTED_OUTPUT = [
    [-2, 4, -8, 14, -4, 0, -42, 48, -4, 2],
    [2, 0, 48, 10, -20, 4, 2, -16, 4, -14],
    [-16, 42, 2, 4, -2, -14, 8, 2, 14, -8],
    [50, -8, -8, -10, -12, 16, -2, 0, -8, -2],
    [-6, -16, -12, -10, 40, -12, 10, 8, -12, 22],
    [-24, 30, -2, 4, -2, -18, 0, 6, 14, 12],
    [-12, -10, -26, 0, 38, 6, -4, -6, 14, 8],
    [-20, -2, -18, 4, 10, 2, -8, 6, 2, 44],
    [-2, 0, 4, -22, 4, 16, 26, -28, 12, 2],
    [-6, -12, -20, 6, 12, -24, -10, 28, -4, 34]
]

In [ ]:
# Run all test cases
for test_num, test_inputs in enumerate(INPUT_BITS):
    print(f"\nTest Case {test_num + 1}:")
    
    # Write inputs
    for i in range(25):
        addr = 0x10 + (i * 8)
        bnn.write(addr, test_inputs[i])
    
    # Start
    bnn.write(0x00, 0x01)
    
    # Wait for done
    while not (bnn.read(0x00) & 0x02):
        time.sleep(0.001)
    
    # Read outputs
    outputs = []
    for i in range(10):
        val = bnn.read(0x100 + i * 4)
        # Convert unsigned 32-bit to signed 32-bit
        if val >= 2**31:
            val = val - 2**32
        outputs.append(val)
    
    # Check if outputs match expected
    if outputs == EXPECTED_OUTPUT[test_num]:
        print(f"PASS")
    else:
        print(f"FAIL")
        print(f"Expected: {EXPECTED_OUTPUT[test_num]}")
        print(f"Got: {outputs}")